In [23]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../../") / "data" / "laptop_data.csv"
df = pd.read_csv(RAW_PATH, encoding="ISO-8859-1")
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,"71,378.68"
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,"47,895.52"
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,"30,636.00"
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,"135,195.34"
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,"96,095.81"


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   TypeName          1303 non-null   object 
 3   Inches            1303 non-null   float64
 4   ScreenResolution  1303 non-null   object 
 5   Cpu               1303 non-null   object 
 6   Ram               1303 non-null   object 
 7   Memory            1303 non-null   object 
 8   Gpu               1303 non-null   object 
 9   OpSys             1303 non-null   object 
 10  Weight            1303 non-null   object 
 11  Price             1303 non-null   float64
dtypes: float64(2), int64(1), object(9)
memory usage: 122.3+ KB


In [25]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,"1,303.00",NaN,NaN,NaN,651.00,376.29,0.00,325.50,651.00,976.50,"1,302.00"
Company,1303,19,Dell,297,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TypeName,1303,6,Notebook,727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Inches,"1,303.00",NaN,NaN,NaN,15.02,1.43,10.10,14.00,15.60,15.60,18.40
ScreenResolution,1303,40,Full HD 1920x1080,507,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cpu,1303,118,Intel Core i5 7200U 2.5GHz,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ram,1303,9,8GB,619,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Memory,1303,39,256GB SSD,412,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gpu,1303,110,Intel HD Graphics 620,281,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OpSys,1303,9,Windows 10,1072,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Data cleaning and preprocessing steps

df.drop(columns='Unnamed: 0', inplace=True)


# 1. Check for missing values
print("\nMissing values before cleaning:")
df.isnull().sum()

# 2. Clean Weight column
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)

# 3. Clean RAM column
df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)

# 4. Extract CPU Speed
def extract_cpu_speed(cpu_str):
    try:
        # First try standard pattern
        speed = float(cpu_str.split('GHz')[0].split()[-1])
    except:
        try:
            # Try alternative patterns
            if 'MHz' in cpu_str:
                speed = float(cpu_str.split('MHz')[0].split()[-1]) / 1000
            else:
                # If no GHz/MHz found, use median
                speeds = df['Cpu'].str.extract(r'(\d+\.?\d*)\s?GHz')[0].dropna().astype(float)
                speed = speeds.median()
        except:
            speed = 2.5  # Default fallback
    return speed

df['Cpu_Speed'] = df['Cpu'].apply(extract_cpu_speed)

# 5. Process Memory into SSD/HDD
def process_memory(mem):
    ssd = hdd = 0
    mem = str(mem)  # Ensure string type
    
    # Handle TB conversions
    if 'TB' in mem:
        multiplier = 1000
    else:
        multiplier = 1
    
    if 'SSD' in mem or 'Flash Storage' in mem:
        size = ''.join(filter(str.isdigit, mem.split('GB')[0]))
        ssd = int(size) * multiplier if size else 0
    elif 'HDD' in mem:
        size = ''.join(filter(str.isdigit, mem.split('GB')[0]))
        hdd = int(size) * multiplier if size else 0
    elif '+' in mem:  # Hybrid case
        parts = mem.split('+')
        for part in parts:
            if 'SSD' in part:
                size = ''.join(filter(str.isdigit, part.split('GB')[0]))
                ssd = int(size) * multiplier if size else 0
            elif 'HDD' in part:
                size = ''.join(filter(str.isdigit, part.split('GB')[0]))
                hdd = int(size) * multiplier if size else 0
    return ssd, hdd

df['SSD'] = df['Memory'].apply(lambda x: process_memory(x)[0])
df['HDD'] = df['Memory'].apply(lambda x: process_memory(x)[1])

# 6. Screen Features
df['Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS').astype(int)

# Extract resolution with error handling
def get_resolution(res_str):
    try:
        res = res_str.split(' ')[-1].split('x')
        return float(res[0]), float(res[1])
    except:
        return 1920.0, 1080.0  

df[['Resolution_X', 'Resolution_Y']] = df['ScreenResolution'].apply(
    lambda x: pd.Series(get_resolution(x)))

# 7. GPU 
df['Gpu_Brand'] = df['Gpu'].str.split().str[0]

# 8. Verify no missing values remain
print("\nMissing values after cleaning:")
df.isnull().sum()

# 9. Check cleaned data
print("\nCleaned data sample:")
df[['Company', 'Ram', 'Cpu_Speed', 'Gpu_Brand', 'SSD', 'HDD', 'Resolution_X', 'Resolution_Y']].head()




Missing values before cleaning:

Missing values after cleaning:

Cleaned data sample:


,Company,Ram,Cpu_Speed,Gpu_Brand,SSD,HDD,Resolution_X,Resolution_Y
0,Apple,8,2.30,Intel,128,0,"2,560.00","1,600.00"
1,Apple,8,1.80,Intel,128,0,"1,440.00",900.00
2,HP,8,2.50,Intel,256,0,"1,920.00","1,080.00"
3,Apple,16,2.70,AMD,512,0,"2,880.00","1,800.00"
4,Apple,8,3.10,Intel,256,0,"2,560.00","1,600.00"


In [27]:
import requests

def get_exchange_rate():
    response = requests.get('https://api.exchangerate-api.com/v4/latest/INR')
    data = response.json()
    return data['rates']['LKR']

# Get current exchange rate
exchange_rate = get_exchange_rate()
print(f"Current exchange rate: 1 INR = {exchange_rate} LKR")

# Convert the Price column
df['Price'] = df['Price'] * exchange_rate
pd.set_option('display.float_format', '{:,.2f}'.format)

df.head()

Current exchange rate: 1 INR = 3.5 LKR


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Cpu_Speed,SSD,HDD,Touchscreen,IPS,Resolution_X,Resolution_Y,Gpu_Brand
0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,"249,825.39",2.30,128,0,0,1,"2,560.00","1,600.00",Intel
1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,"167,634.33",1.80,128,0,0,0,"1,440.00",900.00,Intel
2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,"107,226.00",2.50,256,0,0,0,"1,920.00","1,080.00",Intel
3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,"473,183.68",2.70,512,0,0,1,"2,880.00","1,800.00",AMD
4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,"336,335.33",3.10,256,0,0,1,"2,560.00","1,600.00",Intel


In [28]:
# Save the cleaned DataFrame
CLEANED_PATH = Path("../../") / "data" / "cleaned_data.csv"
df.to_csv(CLEANED_PATH, index=False)